# Benchmark: L2 Correction vs Initial Jdet (Barrier Method)

Per-pixel correlation between initial Jacobian determinant and correction
magnitude for the barrier solver. Unlike the SLSQP version, only the
default Jdet constraint is used.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from dvfopt import jacobian_det2D, generate_random_dvf, scale_dvf
from dvfopt.core.iterative2d_barrier import iterative_2d_barrier
from test_cases import SYNTHETIC_CASES, RANDOM_DVF_CASES, make_deformation, make_random_dvf
from benchmark_utils import run_correction
from benchmark_utils import get_output_dir, save_figure, save_results_csv, save_summary_json, log_run_header, log_run_footer, results_to_rows, show_and_save, reset_figure_counter


In [ ]:
METHOD = "barrier"
NOTEBOOK_NAME = "l2-jdet-correlation"
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base="../output")
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)


## Run barrier on test cases

In [ ]:
CASES = [
    ("01a_10x10_crossing", "10x10 crossing"),
    ("01c_20x40_edges", "20x40 edges"),
    ("03a_10x10_opposite", "10x10 opposite"),
    ("03c_20x20_opposite", "20x20 opposite"),
]
for name in list(RANDOM_DVF_CASES.keys())[:4]:
    CASES.append((name, name))

results = {}
for case_key, label in CASES:
    try:
        dvf, *_ = make_deformation(case_key)
    except Exception:
        dvf = make_random_dvf(case_key)
    phi_init = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac = jacobian_det2D(phi_init)
    if (jac <= 0).sum() == 0:
        print(f"  {label}: no negatives, skipping")
        continue
    r = run_correction(dvf, iterative_2d_barrier)
    results[label] = r
    print(f"  {label}: neg {r['n_neg_init']}->{r['n_neg_final']}  "
          f"min {r['min_jdet_init']:+.4f}->{r['min_jdet']:+.6f}  "
          f"L2={r['l2_err']:.4f}  t={r['time']:.2f}s")


## Per-pixel Jdet vs correction magnitude

In [ ]:
ncols = min(4, len(results))
fig, axes = plt.subplots(2, ncols, figsize=(4*ncols, 8))
if ncols == 1: axes = axes[:, np.newaxis]
for i, (label, r) in enumerate(results.items()):
    if i >= ncols: break
    phi_init = r["phi_init"]; phi = r["phi"]
    jac_init = r["jac_init"][0]
    diff_mag = np.sqrt(((phi - phi_init)**2).sum(axis=0))
    ax = axes[0, i]
    ax.hexbin(jac_init.ravel(), diff_mag.ravel(), gridsize=30, cmap="YlOrRd", mincnt=1)
    rho, _ = stats.spearmanr(jac_init.ravel(), diff_mag.ravel())
    ax.set_title(f"{label}\nSpearman={rho:.3f}", fontsize=9)
    ax.set_xlabel("Init Jdet"); ax.set_ylabel("|correction|")
    ax.axvline(0, color="black", linewidth=0.5, linestyle="--")
    ax2 = axes[1, i]
    ax2.imshow(jac_init, cmap="RdBu_r", origin="upper")
    ax2.contour(diff_mag, levels=5, colors="black", linewidths=0.5)
    ax2.set_title("Jdet + correction contours", fontsize=9)
    ax2.set_xticks([]); ax2.set_yticks([])
plt.suptitle("Per-pixel Jdet vs Correction (Barrier)", fontsize=13)
plt.tight_layout(); show_and_save(OUTPUT_DIR)


## Case-level: min initial Jdet vs total L2

In [ ]:
min_jdets = [r["min_jdet_init"] for r in results.values()]
l2s = [r["l2_err"] for r in results.values()]
labels = list(results.keys())
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(min_jdets, l2s, s=80, edgecolors="black", zorder=3)
for x, y, lab in zip(min_jdets, l2s, labels):
    ax.annotate(lab, (x, y), fontsize=7, textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("Min initial Jdet"); ax.set_ylabel("Total L2 correction")
ax.set_title("L2 vs Worst Initial Jdet (Barrier)")
ax.grid(True, alpha=0.3); plt.tight_layout(); show_and_save(OUTPUT_DIR)


In [ ]:
# --- Save results ---
if 'results' in dir() and isinstance(results, dict) and results:
    rows, cols = results_to_rows(results)
    save_results_csv(rows, cols, OUTPUT_DIR)
    summary = log_run_footer(summary, results)
    save_summary_json(summary, OUTPUT_DIR)
elif 'mag_results' in dir():
    rows, cols = results_to_rows(mag_results)
    save_results_csv(rows, cols, OUTPUT_DIR, name='results_magnitude')
    if 'density_results' in dir():
        rows2, cols2 = results_to_rows(density_results)
        save_results_csv(rows2, cols2, OUTPUT_DIR, name='results_density')
    combined = {**mag_results, **density_results} if 'density_results' in dir() else mag_results
    summary = log_run_footer(summary, combined)
    save_summary_json(summary, OUTPUT_DIR)
else:
    save_summary_json(summary, OUTPUT_DIR)
    print('No results dict found; saved summary only.')
